# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Javeria-crypto326/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis

One row represents one content item (`content_hash_id`).

### Time window

The search-performance data uses a 90-day window, with separate last-30-day and previous-30-day metrics. The model will use this content-level search performance to identify whether impressions are declining.


In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Feature

* `content_type`
* `main_intent`
* `word_count`
* `content_age_days`
* `days_since_last_update`

### Label / proxy

* `trend_direction` — the outcome used to define whether search impressions are declining.

### Context

* `client_hash_id`
* `content_hash_id`
* `window_start`
* `window_end`

### Excluded

* `query_hash_id` — query-level identifier rather than a content feature.
* `provider_used`
* `model_used`
* `is_deleted`
* `is_published`

These fields are excluded because they are identifiers, operational metadata, or status fields rather than the core predictive features for this lane.

### One trap

A key risk is **future information leakage**. For example, `content_updated_date` can be later than the June 30, 2026 reference date. Using information recorded after the prediction reference date could leak future information into the features. Therefore, date-derived features use June 30, 2026 as the reference date, and updates after that date are treated as unavailable.


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [32]:
dim_content = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
""").df()

fact_query = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
)
""").df()

print("dim_content rows:", len(dim_content))
print("fact_content_query_90d rows:", len(fact_query))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

dim_content rows: 519606
fact_content_query_90d rows: 2414248


In [33]:
print("Unique content items:", dim_content["content_hash_id"].nunique())
print("Total dim_content rows:", len(dim_content))

Unique content items: 519606
Total dim_content rows: 519606


In [34]:
print("Unique content-query pairs:",
      fact_query[["content_hash_id", "query_hash_id"]].drop_duplicates().shape[0])

print("Total fact rows:", len(fact_query))

Unique content-query pairs: 2414248
Total fact rows: 2414248


In [35]:
print("Window start:", fact_query["window_start"].min())
print("Window end:", fact_query["window_end"].max())

Window start: 2026-04-02 00:00:00
Window end: 2026-06-30 00:00:00


In [36]:
print("Missing values in selected features:")

for col in ["content_type", "main_intent", "word_count"]:
    print(col, ":", dim_content[col].isna().sum())

print("\nMissing values in label-related fields:")

for col in ["impressions_last30", "impressions_prev30"]:
    print(col, ":", fact_query[col].isna().sum())

Missing values in selected features:
content_type : 0
main_intent : 148398
word_count : 177768

Missing values in label-related fields:
impressions_last30 : 0
impressions_prev30 : 0


### Verification findings

The warehouse data supports the selected content-level grain: `dim_content` contains 519,606 rows and 519,606 unique `content_hash_id` values. The search-performance data contains both last-30-day and previous-30-day impression fields with no missing values.

The selected content features have some missing values: `main_intent` has 148,398 missing values and `word_count` has 177,768 missing values. These missing values should be handled during feature preparation rather than silently dropping the affected content items.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [37]:
import pandas as pd

feature_frame = dim_content[
    [
        "content_hash_id",
        "content_type",
        "main_intent",
        "word_count",
        "content_created_date",
        "content_updated_date"
    ]
].copy()

reference_date = pd.Timestamp("2026-06-30")

feature_frame["content_age_days"] = (
    reference_date -
    pd.to_datetime(feature_frame["content_created_date"])
).dt.days

feature_frame["days_since_last_update"] = (
    reference_date -
    pd.to_datetime(feature_frame["content_updated_date"])
).dt.days

# Exclude records whose dates fall after the search-performance window
feature_frame = feature_frame[
    feature_frame["content_age_days"] >= 0
].copy()

feature_frame.loc[
    feature_frame["days_since_last_update"] < 0,
    "days_since_last_update"
] = pd.NA

feature_frame.head()

,content_hash_id,content_type,main_intent,word_count,content_created_date,content_updated_date,content_age_days,days_since_last_update
0,content_004de9653278b5a4,keyword article,transactional,2555,2026-05-30,2026-07-01,31,NaN
1,content_00dc5efae381b2ab,keyword article,commercial,2430,2026-06-12,2026-07-01,18,NaN
2,content_01410f2556c327ac,keyword article,informational,2645,2026-05-09,2026-07-01,52,NaN
3,content_019f27f634053ca7,keyword article,transactional,2522,2026-06-15,2026-06-15,15,15.0
4,content_01efa71faea45dcc,keyword article,transactional,2552,2026-05-21,2026-06-01,40,29.0


### Feature frame

The feature frame contains a maximum of five candidate features:

* `content_type`
* `main_intent`
* `word_count`
* `content_age_days`
* `days_since_last_update`

`content_age_days` and `days_since_last_update` are derived from the content creation/update dates using June 30, 2026 as the reference date, matching the end of the search-performance window.

Updates dated after the reference date are treated as unavailable rather than using future information.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.